# Content-based baseline: TF-IDF

Items are represented by TF-IDF vectors of `content_text_basic` from `data/items.csv` (warm candidates only). The setup follows the Popularity and MF notebooks: fit on train only, recommend from warm candidates, drop items the user already saw in train, and count validation ratings >= 4 as relevant.

In [1]:
import sys
from pathlib import Path
import importlib.util
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from data_loader import load_data, relevant_items_val

# Load src/content_based.py directly so a stale module is not reused.
content_based_path = ROOT / 'src' / 'content_based.py'
spec = importlib.util.spec_from_file_location('project_content_based', content_based_path)
content_based = importlib.util.module_from_spec(spec)
spec.loader.exec_module(content_based)
TfidfContentRecommender = content_based.TfidfContentRecommender
load_item_text = content_based.load_item_text

train, val, cand_set, seen = load_data(str(ROOT / 'data') + '/')
relevant = relevant_items_val(val, cand_set, seen, min_rating=4)
print(f'train={len(train):,}; validation={len(val):,}; candidates={len(cand_set):,}; evaluable users={len(relevant):,}')

C:\Users\msiba\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


train=427,957; validation=41,284; candidates=22,753; evaluable users=11,850


## Item representation

One document per warm candidate. English stop words are removed; `min_df=2`, `max_df=0.8`.

In [2]:
ITEMS_PATH = ROOT / 'data' / 'items.csv'
TEXT_COL = 'content_text_basic'

item_text = load_item_text(ITEMS_PATH, cand_set, text_col=TEXT_COL)
example_item = next(iter(item_text))
print(f'items with text: {len(item_text):,} / {len(cand_set):,} candidates')
print(f'example item: {example_item}; {item_text[example_item][:150]}...')

items with text: 22,753 / 22,753 candidates
example item: 0000098906; Guiros Wooden Percussion 3 Piece Set 3 Inch Wooden Frog, 4 Inch Wooden Cricket and 3.5 Inches Wooden Owl. Musical Instruments WADSUWAN SHOP Musical In...


## Fit model

The user profile is the mean TF-IDF vector of items the user rated >= 4 in train, and candidates are ranked by cosine similarity to it. Users without a profile, and any empty slots, fall back to popularity.

In [3]:
model = TfidfContentRecommender(
    stop_words='english', min_df=2, max_df=0.8,
    sublinear_tf=True, min_rating=4,
).fit(train, cand_set, item_text)
print(f'Fit complete: {len(model.item_index):,} items, vocab={len(model.vectorizer.vocabulary_):,} terms, users with profile={len(model.user_items):,}')

Fit complete: 22,753 items, vocab=26,550 terms, users with profile=54,036


## Evaluate ranking

Recall@10 and NDCG@10, averaged over users with at least one relevant validation item.

In [4]:
def evaluate_at_k(model, relevant, seen, k=10):
    recalls, ndcgs = [], []
    for user_id, relevant_items in relevant.items():
        recs = model.recommend(user_id, seen, k=k)
        hits = [item in relevant_items for item in recs]
        recalls.append(sum(hits) / len(relevant_items))
        dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal_hits = min(len(relevant_items), k)
        idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
        ndcgs.append(dcg / idcg if idcg else 0.0)
    return {f'Recall@{k}': float(np.mean(recalls)), f'NDCG@{k}': float(np.mean(ndcgs))}

metrics = evaluate_at_k(model, relevant, seen, k=10)
metrics

{'Recall@10': 0.012722345528611938, 'NDCG@10': 0.007451681070760173}

## Inspect recommendations

Recommendations for one evaluable user.

In [5]:
if relevant:
    example_user = next(iter(relevant))
    print('user:', example_user)
    print('recommendations:', model.recommend(example_user, seen, k=10))
    print('relevant validation items:', relevant[example_user])
else:
    print('No evaluable users in validation.')

user: AE22B7BMXT7GAGY3XLHICMESNAPA
recommendations: ['B086QKG5WZ', 'B07DK964NQ', 'B07P1CCFFS', 'B07QKC1RND', 'B00HI4CULS', 'B07T9FHGWG', 'B00SXWIG8A', 'B01H9DB18K', 'B00W3IFCG4', 'B00KCXL09A']
relevant validation items: {'B0BYNM15YB'}


## Save outputs

Writes configs, metrics, recs, logs and samples to `outputs/` with fixed file names that are overwritten on every run; the run date and time are recorded in the config and log. The saved metrics are checked against `evaluate_at_k`.

In [ ]:
import json
import platform
import time
from datetime import datetime

import pandas as pd
import scipy
import sklearn

K = 10
SEED = 42  # only used to pick sample users
OUT = ROOT / 'outputs'
run_started = datetime.now()
run_id = 'tfidf_cbf_' + run_started.strftime('%Y%m%d-%H%M%S')
OUTPUT_FILES = {
    'configs': OUT / 'configs' / 'tfidf_cbf.json',
    'metrics': OUT / 'metrics' / 'tfidf_cbf_val.json',
    'recs': OUT / 'recs' / 'tfidf_cbf_val_top10.csv',
    'logs': OUT / 'logs' / 'tfidf_cbf_run.json',
    'samples': OUT / 'samples' / 'tfidf_cbf_examples.md',
    'figures': OUT / 'figures' / 'tfidf_cbf_metrics.png',
}
for file_path in OUTPUT_FILES.values():
    file_path.parent.mkdir(parents=True, exist_ok=True)

def rel_path(p):
    return Path(p).relative_to(ROOT).as_posix()

# Refit only for timing; vocabulary must match.
t0 = time.perf_counter()
timing_model = TfidfContentRecommender(
    stop_words=model.stop_words, min_df=model.min_df, max_df=model.max_df,
    sublinear_tf=model.sublinear_tf, min_rating=model.min_rating,
).fit(train, cand_set, item_text)
fit_seconds = time.perf_counter() - t0
assert timing_model.vectorizer.vocabulary_ == model.vectorizer.vocabulary_, 'refit vocabulary differs'
del timing_model

# Recs, scores and metrics per evaluable user.
t0 = time.perf_counter()
rows, recalls, ndcgs, hit_rates = [], [], [], []
no_profile_users, partial_fill_users = [], []
for user_id, relevant_items in relevant.items():
    recs = model.recommend(user_id, seen, k=K)
    profile = model.user_profile(user_id)
    if profile is None:
        no_profile_users.append(user_id)
        scores = [np.nan] * len(recs)
    else:
        cand_scores = model.cand_matrix @ profile
        scores = [float(cand_scores[model.cand_pos[item]]) for item in recs]
        if any(s <= 0 for s in scores):
            partial_fill_users.append(user_id)
    hits = [item in relevant_items for item in recs]
    recalls.append(sum(hits) / len(relevant_items))
    dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
    ideal_hits = min(len(relevant_items), K)
    idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
    ndcgs.append(dcg / idcg if idcg else 0.0)
    hit_rates.append(float(any(hits)))
    rows.extend(
        {'user_id': user_id, 'item_id': item, 'score': score, 'rank': rank + 1}
        for rank, (item, score) in enumerate(zip(recs, scores))
    )
recs_eval_seconds = time.perf_counter() - t0

saved_metrics = {
    f'Recall@{K}': float(np.mean(recalls)),
    f'NDCG@{K}': float(np.mean(ndcgs)),
    f'HitRate@{K}': float(np.mean(hit_rates)),
    'n_eval_users': len(relevant),
    'n_eval_users_with_profile': len(relevant) - len(no_profile_users),
}
assert saved_metrics[f'Recall@{K}'] == metrics[f'Recall@{K}'], 'Recall differs from evaluate_at_k'
assert saved_metrics[f'NDCG@{K}'] == metrics[f'NDCG@{K}'], 'NDCG differs from evaluate_at_k'

recs_df = pd.DataFrame(rows, columns=['user_id', 'item_id', 'score', 'rank'])
recs_path = OUTPUT_FILES['recs']
recs_df.to_csv(recs_path, index=False)

vectorizer_params = {
    k: v for k, v in model.vectorizer.get_params().items()
    if isinstance(v, (str, int, float, bool, tuple, type(None)))
}
config = {
    'run_id': run_id,
    'run_started': run_started.isoformat(timespec='seconds'),
    'model': 'TfidfContentRecommender',
    'text_source': {
        'path': rel_path(ITEMS_PATH),
        'column': TEXT_COL,
        'filter': 'item_id in data/candidate_items_warm.csv',
        'n_items_with_text': len(item_text),
    },
    'params': {
        'stop_words': model.stop_words, 'min_df': model.min_df, 'max_df': model.max_df,
        'sublinear_tf': model.sublinear_tf, 'min_rating': model.min_rating, 'k': K,
    },
    'vectorizer_params': vectorizer_params,
    'user_profile': 'mean TF-IDF vector of items rated >= min_rating in data/interactions_train.csv',
    'scoring': 'cosine similarity (L2-normalized item vectors); train-seen items excluded; top-k',
    'fallback': 'popularity (train rating >= min_rating) for users without a profile and for zero-score slots',
    'seed': SEED,
    'seed_note': 'The TF-IDF model is deterministic; the seed is only used to pick the 3 sample users in samples/.',
    'data': {
        'train': 'data/interactions_train.csv',
        'validation': 'data/interactions_validation.csv',
        'candidates': 'data/candidate_items_warm.csv',
        'items': rel_path(ITEMS_PATH),
    },
    'environment': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': scipy.__version__,
        'scikit-learn': sklearn.__version__,
    },
}

# Pick sample users (seeded).
titles = pd.read_csv(ITEMS_PATH, dtype={'item_id': str}, usecols=['item_id', 'title']).set_index('item_id')['title']
item_stats = train.groupby('item_id')['rating'].agg(['size', 'mean'])
rng = np.random.default_rng(SEED)
pool = sorted(u for u in relevant if u in model.user_items)
sample_users = rng.choice(pool, size=min(3, len(pool)), replace=False).tolist()

def md_title(item):
    title = titles.get(item)
    return '(no title)' if pd.isna(title) else str(title).replace('|', '\\|')

def md_stats(item):
    if item not in item_stats.index:
        return '0 | -'
    n, mean = item_stats.loc[item]
    return f'{int(n)} | {mean:.2f}'

MAX_LIKED = 10
lines = [
    '# TF-IDF CBF sample recommendations', '',
    f'Item text: `{TEXT_COL}` from `{rel_path(ITEMS_PATH)}`. 3 users picked at random (seed={SEED}) from evaluable users with a profile.',
    'Train interactions and mean train rating are computed over all users in `interactions_train.csv`.', '',
]
for user_id in sample_users:
    liked = train[(train['user_id'] == user_id) & (train['rating'] >= model.min_rating)]
    recs = recs_df[recs_df['user_id'] == user_id]
    n_hits = int(recs['item_id'].isin(relevant[user_id]).sum())
    lines += [f'## User `{user_id}`', '',
              f'Validation hits: {n_hits} of {len(relevant[user_id])} relevant items.', '',
              f'### Liked in train (rating >= {model.min_rating}, {len(liked)} items)', '',
              '| item_id | title | user rating | train interactions | mean train rating |',
              '|---|---|---|---|---|']
    for row in liked.head(MAX_LIKED).itertuples():
        lines.append(f'| {row.item_id} | {md_title(row.item_id)} | {row.rating:.0f} | {md_stats(row.item_id)} |')
    if len(liked) > MAX_LIKED:
        lines.append(f'| ... | (+{len(liked) - MAX_LIKED} more) | | | |')
    lines += ['', '### Top-10 recommendations', '',
              '| rank | item_id | title | score | train interactions | mean train rating | relevant in validation |',
              '|---|---|---|---|---|---|---|']
    for row in recs.itertuples():
        hit = 'yes' if row.item_id in relevant[user_id] else ''
        lines.append(f'| {row.rank} | {row.item_id} | {md_title(row.item_id)} | {row.score:.4f} | {md_stats(row.item_id)} | {hit} |')
    lines.append('')
OUTPUT_FILES['samples'].write_text('\n'.join(lines), encoding='utf-8')

run_finished = datetime.now()
log = {
    'run_id': run_id,
    'run_started': run_started.isoformat(timespec='seconds'),
    'run_finished': run_finished.isoformat(timespec='seconds'),
    'fit_seconds': round(fit_seconds, 2),
    'recs_eval_seconds': round(recs_eval_seconds, 2),
    'output_stage_seconds': round((run_finished - run_started).total_seconds(), 2),
    'vocab_size': len(model.vectorizer.vocabulary_),
    'n_items_with_text': len(item_text),
    'n_eval_users': len(relevant),
    'fallback': {
        'no_profile_full_popularity': len(no_profile_users),
        'partial_popularity_fill': len(partial_fill_users),
    },
    'checks': {
        'refit_vocabulary_identical': True,
        'metrics_match_evaluate_at_k': True,
    },
    'notes': [
        'fit_seconds comes from a refit with the same parameters; its vocabulary is checked against the notebook model.',
        'Earlier finding (interaction-based TF-IDF, commit 47cc565): NDCG@10 differed slightly between environments, '
        '0.005738 (eda_env, scikit-learn 1.9.1) vs 0.005732 (Anaconda, scikit-learn 1.3.0), probably due to the '
        'order of tied scores. Ranking tie-breaking has not been changed.',
    ],
}

for key, payload in [('configs', config), ('metrics', saved_metrics), ('logs', log)]:
    OUTPUT_FILES[key].write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

print('run_id:', run_id)
print('metrics:', saved_metrics)
print(f"fallback users: {len(no_profile_users)} without profile, {len(partial_fill_users)} partially filled by popularity")
print('files:')
for key in ['configs', 'metrics', 'recs', 'logs', 'samples']:
    print(' ', rel_path(OUTPUT_FILES[key]))

## Model comparison

Popularity and MF use the same classes and parameters as their own notebooks and are evaluated on the same users. HitRate@10 is the share of users with at least one relevant item in the top 10. MF takes about 5 minutes to fit.

In [ ]:
from IPython.display import Markdown, display
from popularity import PopularityRecommender
from matrix_factorization import MatrixFactorizationRecommender

def hit_rate_at_k(model, relevant, seen, k=10):
    return float(np.mean([
        any(item in relevant_items for item in model.recommend(user_id, seen, k=k))
        for user_id, relevant_items in relevant.items()
    ]))

pop_model = PopularityRecommender(min_rating=4).fit(train, cand_set)
mf_model = MatrixFactorizationRecommender(
    n_factors=32, n_epochs=10, learning_rate=0.01,
    regularization=0.02, min_rating=4, random_state=42,
).fit(train, cand_set)

comparison = pd.DataFrame(
    [
        {**evaluate_at_k(pop_model, relevant, seen, k=K), f'HitRate@{K}': hit_rate_at_k(pop_model, relevant, seen, k=K)},
        {**evaluate_at_k(mf_model, relevant, seen, k=K), f'HitRate@{K}': hit_rate_at_k(mf_model, relevant, seen, k=K)},
        {key: saved_metrics[key] for key in [f'Recall@{K}', f'NDCG@{K}', f'HitRate@{K}']},
    ],
    index=pd.Index(['Popularity', 'MF', 'TF-IDF CBF'], name='model'),
)
comparison['n_eval_users'] = len(relevant)
display(comparison.style.format({f'Recall@{K}': '{:.5f}', f'NDCG@{K}': '{:.5f}', f'HitRate@{K}': '{:.5f}', 'n_eval_users': '{:,}'}))

### Sanity-check subset

The same 200 users as `notebooks/sanity_check_topn.ipynb` (`random.seed(42)`), with the same rule check: exactly 10 unique items, none seen in train, all warm candidates.

In [ ]:
import random

random.seed(42)
sanity_users = random.sample(sorted(relevant), 200)
sanity_relevant = {u: relevant[u] for u in sanity_users}

def rule_violations(model, users, k=10):
    count = 0
    for u in users:
        recs = model.recommend(u, seen, k=k)
        ok = (len(recs) == k and len(set(recs)) == k
              and not (set(recs) & seen[u]) and set(recs) <= cand_set)
        count += not ok
    return count

sanity = pd.DataFrame(
    [
        {**evaluate_at_k(m, sanity_relevant, seen, k=K),
         f'HitRate@{K}': hit_rate_at_k(m, sanity_relevant, seen, k=K),
         'rule violations': f'{rule_violations(m, sanity_users, K)}/{len(sanity_users)}'}
        for m in [pop_model, mf_model, model]
    ],
    index=pd.Index(['Popularity', 'MF', 'TF-IDF CBF'], name='model'),
)
display(sanity.style.format({f'Recall@{K}': '{:.4f}', f'NDCG@{K}': '{:.4f}', f'HitRate@{K}': '{:.4f}'}))

## Example recommendations

The same 3 users as in `outputs/samples/` (picked at random, seed 42). Item stats are computed over all train users.

In [ ]:
def with_item_info(df):
    out = df.copy()
    out.insert(out.columns.get_loc('item_id') + 1, 'title', out['item_id'].map(titles))
    out['train_interactions'] = out['item_id'].map(item_stats['size']).fillna(0).astype(int)
    out['train_mean_rating'] = out['item_id'].map(item_stats['mean']).round(2)
    return out

for user_id in sample_users:
    liked = train.loc[(train['user_id'] == user_id) & (train['rating'] >= model.min_rating), ['item_id', 'rating']]
    user_recs = recs_df.loc[recs_df['user_id'] == user_id, ['rank', 'item_id', 'score']]
    user_recs = with_item_info(user_recs)
    user_recs['relevant_in_validation'] = user_recs['item_id'].isin(relevant[user_id])
    display(Markdown(
        f'### User `{user_id}`\n'
        f'{len(liked)} items liked in train; validation hits: '
        f'{int(user_recs["relevant_in_validation"].sum())} of {len(relevant[user_id])} relevant items.'
    ))
    display(Markdown('**Liked in train (rating >= 4)**'))
    display(with_item_info(liked).reset_index(drop=True))
    display(Markdown('**Top-10 TF-IDF recommendations**'))
    display(user_recs.style.hide(axis='index').format({'score': '{:.4f}', 'train_mean_rating': '{:.2f}'}))

## TF-IDF model summary

In [ ]:
model_summary = pd.DataFrame(
    {
        'value': [
            len(model.candidate_items),
            len(model.item_index),
            len(model.vectorizer.vocabulary_),
            len(relevant),
            len(no_profile_users),
            len(partial_fill_users),
            TEXT_COL,
        ]
    },
    index=pd.Index([
        'warm candidate items',
        'items with TF-IDF vector',
        'vocabulary size',
        'evaluable users',
        'fallback users: no profile (full popularity)',
        'fallback users: partial popularity fill',
        'text column',
    ], name='item'),
)
display(model_summary)

vectorizer_table = pd.DataFrame(
    {'value': [str(v) for v in vectorizer_params.values()]},
    index=pd.Index(list(vectorizer_params), name='TfidfVectorizer parameter'),
)
display(vectorizer_table)

## Recall@10 and NDCG@10

Also saved to `outputs/figures/tfidf_cbf_metrics.png`.

In [ ]:
import matplotlib.pyplot as plt

MODEL_COLORS = {'Popularity': '#2a78d6', 'MF': '#eb6834', 'TF-IDF CBF': '#1baf7a'}
INK, MUTED, GRID, BASELINE, SURFACE = '#0b0b0b', '#52514e', '#e1e0d9', '#c3c2b7', '#fcfcfb'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), facecolor=SURFACE)
for ax, metric in zip(axes, [f'Recall@{K}', f'NDCG@{K}']):
    values = comparison[metric]
    bars = ax.bar(values.index, values.values, width=0.55,
                  color=[MODEL_COLORS[m] for m in values.index], edgecolor=SURFACE, linewidth=2, zorder=2)
    ax.bar_label(bars, labels=[f'{v:.4f}' for v in values.values], padding=3, fontsize=9, color=INK)
    ax.set_title(metric, loc='left', fontsize=11, color=INK)
    ax.set_ylim(0, values.max() * 1.18)
    ax.set_facecolor(SURFACE)
    ax.grid(axis='y', color=GRID, linewidth=0.8, zorder=0)
    ax.tick_params(colors=MUTED, labelsize=9, length=0)
    for side in ['top', 'right', 'left']:
        ax.spines[side].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE)
fig.suptitle(f'TF-IDF CBF vs baselines: {len(relevant):,} evaluable users, k={K}',
             x=0.01, ha='left', fontsize=12, color=INK)
fig.tight_layout()

figure_path = OUTPUT_FILES['figures']
fig.savefig(figure_path, dpi=150, facecolor=SURFACE, bbox_inches='tight')
plt.show()
display(Markdown(f'Saved to `{rel_path(figure_path)}`'))